# BME 507 — Warm-up Exercise 7: Organ Localization

**Goal:** localize the **heart, right lung, and left lung** with bounding boxes.

This notebook deliberately keeps Task 7 simple and reuses the best Task 6 segmentation model:

**X-ray → Task 6 segmentation mask → bounding boxes**

For each organ, the notebook derives:
- ground-truth box from the ground-truth segmentation mask
- predicted box from the predicted segmentation mask
- bounding-box IoU

It saves a per-image CSV, summary CSV, an organ-IoU figure, and qualitative examples.

In [ ]:
# ============================================================
# BME 507 - WARM-UP EXERCISE 7
# Organ localization from Task 6 segmentation
# Heart + right lung + left lung bounding boxes
# ============================================================

import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from google.colab import drive

# ------------------------------------------------------------
# 1. SETTINGS + GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")

SAVE_DIR = Path("/content/drive/MyDrive/BME507/Warmup_5_6_7")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = SAVE_DIR / "task6_best_model.pth"

ZIP_PATH = Path("/content/Segmentation02.zip")
EXTRACT_DIR = Path("/content/task7_segmentation02")

DATA_URLS = [
    "https://imgcom.jsrt.or.jp/imgcom/wp-content/uploads/2018/11/Segmentation02.zip",
    "http://imgcom.jsrt.or.jp/imgcom/wp-content/uploads/2018/11/Segmentation02.zip",
]

IMAGE_SIZE = 256
BATCH_SIZE = 8

CLASS_NAMES = [
    "outside_body",
    "chest_area",
    "heart",
    "right_lung",
    "left_lung",
]

ORGAN_IDS = {
    "heart": 2,
    "right_lung": 3,
    "left_lung": 4,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"{MODEL_PATH} was not found. Run Warm-up Exercise 6 first."
    )

# ------------------------------------------------------------
# 2. GET + EXTRACT SEGMENTATION02
# ------------------------------------------------------------

def download_if_needed(zip_path, urls):
    if zip_path.exists() and zip_path.stat().st_size > 1000:
        print("Dataset zip already exists:", zip_path)
        return

    for url in urls:
        print("Trying:", url)
        result = subprocess.run(
            ["wget", "-q", "-O", str(zip_path), url],
            capture_output=True
        )
        if result.returncode == 0 and zip_path.exists() and zip_path.stat().st_size > 1000:
            print("Downloaded:", zip_path)
            return

    raise FileNotFoundError(
        "Could not download Segmentation02.zip automatically. "
        "Upload Segmentation02.zip manually to /content/Segmentation02.zip and rerun."
    )

def find_named_dir(root, name):
    for current, dirs, files in os.walk(root):
        if Path(current).name.lower() == name.lower():
            return Path(current)
    raise FileNotFoundError(f"Folder '{name}' was not found under {root}")

def pair_images_and_masks(image_dir, mask_dir):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

    images = {
        p.stem: p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    }
    masks = {
        p.stem: p for p in mask_dir.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    }

    common = sorted(set(images) & set(masks))
    if not common:
        raise RuntimeError(
            f"No matching image/mask filenames found in:\n{image_dir}\n{mask_dir}"
        )

    return [(images[k], masks[k]) for k in common]

download_if_needed(ZIP_PATH, DATA_URLS)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

org_test = find_named_dir(EXTRACT_DIR, "org_test")
label_test = find_named_dir(EXTRACT_DIR, "label_test")
test_pairs = pair_images_and_masks(org_test, label_test)

print("Official test images:", len(test_pairs))

# ------------------------------------------------------------
# 3. CONVERT MINIJSRT LABEL TO 5 CLASSES
# ------------------------------------------------------------

def split_lung_components(lung_binary):
    lung_u8 = lung_binary.astype(np.uint8)

    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        lung_u8, connectivity=8
    )

    component_ids = list(range(1, n_labels))
    component_ids.sort(
        key=lambda k: stats[k, cv2.CC_STAT_AREA],
        reverse=True
    )
    component_ids = component_ids[:2]

    h, w = lung_u8.shape

    if len(component_ids) < 2:
        x = np.arange(w)[None, :]
        right = lung_u8.astype(bool) & (x < w // 2)
        left = lung_u8.astype(bool) & (x >= w // 2)
        return right, left

    a, b = component_ids
    if centroids[a][0] < centroids[b][0]:
        right_id, left_id = a, b
    else:
        right_id, left_id = b, a

    return labels == right_id, labels == left_id

def raw_mask_to_classes(raw):
    class_mask = np.zeros_like(raw, dtype=np.int64)

    class_mask[raw == 170] = 1
    class_mask[raw == 85] = 2

    lung = raw == 255
    right_lung, left_lung = split_lung_components(lung)

    class_mask[right_lung] = 3
    class_mask[left_lung] = 4

    return class_mask

class OrganDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR
        )
        mask = Image.open(mask_path).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST
        )

        image_np = np.asarray(image, dtype=np.float32) / 255.0
        raw_mask = np.asarray(mask, dtype=np.uint8)
        class_mask = raw_mask_to_classes(raw_mask)

        return (
            torch.from_numpy(image_np).unsqueeze(0),
            torch.from_numpy(class_mask).long(),
            image_path.name,
        )

test_dataset = OrganDataset(test_pairs)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

# ------------------------------------------------------------
# 4. SAME U-NET AS TASK 6
# ------------------------------------------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=5, base=16):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = DoubleConv(base * 16, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)

        self.out = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)

model = UNet(in_channels=1, out_channels=5, base=16).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("Loaded:", MODEL_PATH)

# ------------------------------------------------------------
# 5. BOUNDING BOX FUNCTIONS
# ------------------------------------------------------------

def mask_to_box(mask_binary):
    ys, xs = np.where(mask_binary)

    if len(xs) == 0 or len(ys) == 0:
        return None

    return (
        int(xs.min()),
        int(ys.min()),
        int(xs.max()),
        int(ys.max()),
    )

def box_iou(box_a, box_b):
    if box_a is None or box_b is None:
        return 0.0

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    inter_w = max(0, ix2 - ix1 + 1)
    inter_h = max(0, iy2 - iy1 + 1)
    intersection = inter_w * inter_h

    area_a = (ax2 - ax1 + 1) * (ay2 - ay1 + 1)
    area_b = (bx2 - bx1 + 1) * (by2 - by1 + 1)

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0

# ------------------------------------------------------------
# 6. LOCALIZE ALL TEST IMAGES
# ------------------------------------------------------------

rows = []
saved_predictions = {}

with torch.no_grad():
    for image, gt_mask, filename in test_dataset:
        logits = model(image.unsqueeze(0).to(device))
        pred_mask = logits.argmax(dim=1).cpu().squeeze().numpy()

        gt_np = gt_mask.numpy()
        saved_predictions[filename] = pred_mask

        for organ_name, class_id in ORGAN_IDS.items():
            gt_box = mask_to_box(gt_np == class_id)
            pred_box = mask_to_box(pred_mask == class_id)

            iou = box_iou(gt_box, pred_box)

            row = {
                "filename": filename,
                "organ": organ_name,
                "box_iou": iou,
                "predicted_box_found": pred_box is not None,
            }

            for prefix, box in [("gt", gt_box), ("pred", pred_box)]:
                if box is None:
                    row[f"{prefix}_xmin"] = np.nan
                    row[f"{prefix}_ymin"] = np.nan
                    row[f"{prefix}_xmax"] = np.nan
                    row[f"{prefix}_ymax"] = np.nan
                else:
                    row[f"{prefix}_xmin"] = box[0]
                    row[f"{prefix}_ymin"] = box[1]
                    row[f"{prefix}_xmax"] = box[2]
                    row[f"{prefix}_ymax"] = box[3]

            rows.append(row)

results_df = pd.DataFrame(rows)

summary_df = (
    results_df.groupby("organ")
    .agg(
        mean_box_iou=("box_iou", "mean"),
        std_box_iou=("box_iou", "std"),
        median_box_iou=("box_iou", "median"),
        detection_rate=("predicted_box_found", "mean"),
        n=("box_iou", "size"),
    )
    .reset_index()
)

results_df.to_csv(
    SAVE_DIR / "task7_localization_per_image.csv",
    index=False
)

summary_df.to_csv(
    SAVE_DIR / "task7_localization_summary.csv",
    index=False
)

print("\nTASK 7 SUMMARY")
print(summary_df)

# ------------------------------------------------------------
# 7. SAVE BOX-IoU PLOT
# ------------------------------------------------------------

plt.figure(figsize=(6, 4))
plt.bar(summary_df["organ"], summary_df["mean_box_iou"])
plt.ylim(0, 1)
plt.ylabel("Mean bounding-box IoU")
plt.title("Task 7 - Organ localization")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(
    SAVE_DIR / "task7_box_iou_by_organ.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 8. SAVE QUALITATIVE EXAMPLES
# ------------------------------------------------------------

display_colors = {
    "heart": "tab:red",
    "right_lung": "tab:blue",
    "left_lung": "tab:green",
}

n_examples = min(6, len(test_dataset))
example_indices = np.linspace(0, len(test_dataset) - 1, n_examples, dtype=int)

fig, axes = plt.subplots(
    n_examples,
    1,
    figsize=(6, 5 * n_examples)
)

if n_examples == 1:
    axes = [axes]

for ax, idx in zip(axes, example_indices):
    image, gt_mask, filename = test_dataset[idx]
    image_np = image.squeeze().numpy()
    gt_np = gt_mask.numpy()
    pred_np = saved_predictions[filename]

    ax.imshow(image_np, cmap="gray")
    ax.set_title(
        f"{filename}\nSolid = ground truth, dashed = predicted"
    )
    ax.axis("off")

    for organ_name, class_id in ORGAN_IDS.items():
        gt_box = mask_to_box(gt_np == class_id)
        pred_box = mask_to_box(pred_np == class_id)
        color = display_colors[organ_name]

        if gt_box is not None:
            x1, y1, x2, y2 = gt_box
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1 + 1,
                y2 - y1 + 1,
                linewidth=2,
                edgecolor=color,
                facecolor="none",
                linestyle="-",
                label=f"{organ_name} GT",
            )
            ax.add_patch(rect)

        if pred_box is not None:
            x1, y1, x2, y2 = pred_box
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1 + 1,
                y2 - y1 + 1,
                linewidth=2,
                edgecolor=color,
                facecolor="none",
                linestyle="--",
                label=f"{organ_name} Pred",
            )
            ax.add_patch(rect)

    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="lower right", fontsize=8)

plt.tight_layout()
plt.savefig(
    SAVE_DIR / "task7_examples.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# 9. SAVE RUN INFORMATION
# ------------------------------------------------------------

run_info = [
    f"device={device}",
    f"model_path={MODEL_PATH}",
    f"image_size={IMAGE_SIZE}",
    f"test_images={len(test_pairs)}",
    "organs=heart,right_lung,left_lung",
    "method=bounding boxes extracted from Task 6 predicted segmentation masks",
]

(SAVE_DIR / "task7_run_info.txt").write_text("\n".join(run_info))

print("\nSaved Task 7 outputs to:", SAVE_DIR)